[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/087.%20The%20Hub-and-Spoke%20Network%20Design%20Problem/P87-Tier-3_executed.ipynb)

---



# 87. The Hub-and-Spoke Network Design Problem

## Tier 3: Firefly Algorithm Metaheuristic

This notebook uses a population-based Firefly Algorithm to search for a low-cost hub-and-spoke network. The input is a set of nodes with coordinates, pairwise demands, and candidate hub fixed costs. The output is a selected set of hubs, an assignment of each node to a hub, the associated network cost, and a record of the best cost found over the iterations. A greedy solution is also evaluated as a reference, and selected algorithm parameters are explored through sensitivity experiments.

### Modeling assumptions and scope

- Each node is both a demand origin and destination and a candidate hub location.
- Exactly `max_hubs` hubs are selected. Every selected hub is assigned to itself; each non-hub node is assigned to one selected hub.
- The generated demand matrix is symmetric, but the objective sums over ordered origin-destination pairs. Thus, each nonzero pair contributes once in each direction.
- A shipment from origin `i` to destination `j` follows the route from `i` to its assigned hub, between the two assigned hubs, and from the destination's assigned hub to `j`. Only inter-hub travel receives the discount factor.
- Distances are Euclidean distances between generated coordinates. The model has no capacity, congestion, service-level, or time-window constraints. Cost values are model units, not a specified currency.
- The Firefly Algorithm is a stochastic search heuristic. It can find good feasible solutions, but it does not certify global optimality.

### Search procedure

1. **Represent a solution:** Store hub-selection and assignment information in a continuous position vector with two components per node.
2. **Decode a position:** Select the `max_hubs` largest hub-selection components. Assign each selected hub to itself; map each other node's assignment component to the nearest selected hub index.
3. **Evaluate the network:** Compute fixed hub costs and demand-weighted transportation costs. Lower total cost means a brighter firefly in the algorithm's cost-based comparison.
4. **Compare fireflies:** Measure Euclidean distance between their encoded positions and calculate distance-dependent attractiveness.
5. **Move and evaluate:** Move a higher-cost firefly toward a lower-cost one, add a random perturbation, decode the new position, and evaluate it.
6. **Track the best solution:** Record the lowest-cost solution found so far and its cost after each iteration.

The encoding is a continuous search representation for a discrete network decision: decoded solutions always contain the required number of hubs and a valid hub assignment for every node. Distance between fireflies is measured in this encoded search space, not in the physical network's coordinate space.

### What to examine

- The convergence history shows the best cost found so far; it cannot increase because the best-so-far solution is retained.
- The network plot shows hub locations and assignments. It represents assignments, not the separate origin-destination shipment flows used in the cost calculation.
- The cost breakdown separates fixed hub costs from variable transportation costs.
- The greedy comparison and parameter experiments are empirical results for the specified instance and random search run. They do not establish a general performance guarantee.

## Mathematical model and algorithm

Let the node set be $N = \{0, 1, \ldots, n-1\}$. The model selects exactly $p$ hubs from these nodes and assigns each node to one selected hub. Define $H \subseteq N$ as the selected hub set and $a_i \in H$ as the hub assigned to node $i$. Selected hubs are self-assigned, so $a_i=i$ for every $i \in H$.

For each node pair, let $w_{ij}$ be the demand from origin $i$ to destination $j$, and let $d_{ij}$ be their Euclidean distance. Let $f_k$ be the fixed cost of opening hub $k$, $c$ the transportation cost per unit distance, and $\alpha$ the inter-hub discount factor. The cost of routing one unit from $i$ to $j$ is the collection distance to $a_i$, the discounted distance between hubs, and the distribution distance from $a_j$ to $j$. The objective is

$$\min_{H, a}\quad \sum_{k \in H} f_k + c \sum_{i \in N} \sum_{\substack{j \in N \\ j \ne i}} w_{ij} \left(d_{i,a_i} + \alpha d_{a_i,a_j} + d_{a_j,j}\right)$$

subject to

$$|H| = p, \qquad a_i \in H \quad (i \in N), \qquad a_i=i \quad (i \in H).$$

Here, $n$ is `num_nodes`, $p$ is `max_hubs`, $f_k$ is `hub_costs[k]`, $w_{ij}$ is `demand_matrix[i, j]`, $d_{ij}$ is `distance_matrix[i, j]`, $c$ is `transport_cost`, and $\alpha$ is `alpha`. The implementation sums over ordered pairs with $i \ne j$, matching the demand matrix and the loops in `evaluate_solution`. Because every selected hub assigns itself, its own collection and distribution distances are zero when it is the corresponding endpoint.

### Firefly movement

A firefly's position is a vector $x \in \mathbb{R}^{2n}$, containing $n$ hub-selection components and $n$ assignment components. Its decoded network has cost $C(x)$. For positions $x_i$ and $x_j$, the algorithm uses

$$r_{ij} = \lVert x_i - x_j \rVert_2, \qquad \beta(r_{ij}) = \beta_0 \exp(-\gamma r_{ij}^{2}),$$

$$x_i \leftarrow x_i + \beta(r_{ij})(x_j-x_i) + \alpha_{\mathrm{rand}}\left(u-\frac{1}{2}\right), \qquad u \sim U([0,1]^{2n}),$$

when $C(x_i)>C(x_j)$. Here, $r_{ij}$ is encoded-space distance, $\beta_0$ is attractiveness at zero distance, $\gamma$ controls how quickly attractiveness decreases with distance, and $\alpha_{\mathrm{rand}}$ is the randomization parameter. The random vector provides perturbations that help explore the search space. In the code, these quantities correspond to `calculate_distance`, `calculate_attractiveness`, and `move_firefly`; `alpha_param` is $\alpha_{\mathrm{rand}}$, distinct from the transportation discount `alpha`.

The best-so-far cost starts at the lowest cost in the initial population. After each iteration it is updated only when a lower cost is found, so the convergence history is non-increasing. With population size $P$, $T$ iterations, and an $O(n^2)$ evaluation of the ordered origin-destination pairs, the main search has approximate time complexity $O(TP^2n^2)$. The nested pairwise movement loop compares up to $P(P-1)$ ordered pairs per iteration; each successful move triggers a solution evaluation. The algorithm is a heuristic search over decoded solutions, not an exact optimization procedure.

In [ ]:
from typing import Dict, List, Tuple

import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Use consistent plotting defaults so that network and convergence patterns are easy to inspect.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class FireflyHubSpokeOptimizer:
    """Firefly Algorithm for hub-and-spoke network design."""

    def __init__(self, num_nodes=15, max_hubs=5, population_size=20):
        self.num_nodes = num_nodes
        self.max_hubs = max_hubs
        self.population_size = population_size
        self.alpha = 0.75  # Inter-hub transportation discount factor.

        # Firefly parameters control attraction and random exploration.
        self.beta0 = 1.0
        self.gamma = 0.1
        self.alpha_param = 0.5

        self._initialize_network_data()

    def _initialize_network_data(self):
        """Create reproducible coordinates, demands, distances, and costs."""

        # Reset both random-number generators so that the example is reproducible.
        np.random.seed(42)
        random.seed(42)

        self.coordinates = np.random.uniform(0, 100, (self.num_nodes, 2))

        self.demand_matrix = np.random.randint(
            10, 100, (self.num_nodes, self.num_nodes)
        )
        np.fill_diagonal(self.demand_matrix, 0)
        self.demand_matrix = (
            self.demand_matrix + self.demand_matrix.T
        ) // 2

        self.distance_matrix = np.zeros((self.num_nodes, self.num_nodes))
        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i != j:
                    dx = self.coordinates[i, 0] - self.coordinates[j, 0]
                    dy = self.coordinates[i, 1] - self.coordinates[j, 1]
                    self.distance_matrix[i, j] = np.sqrt(dx**2 + dy**2)

        self.hub_costs = np.random.randint(1000, 5000, self.num_nodes)
        self.transport_cost = 1.0

    def encode_solution(self, hub_selection, spoke_assignments):
        """Encode hub indicators and assigned-hub indices as one position."""

        hub_vector = np.zeros(self.num_nodes)
        for hub in hub_selection:
            hub_vector[hub] = 1.0

        assignment_vector = np.zeros(self.num_nodes)
        for node, hub in spoke_assignments.items():
            assignment_vector[node] = hub

        return np.concatenate([hub_vector, assignment_vector])

    def decode_solution(self, position):
        """Decode a position into exactly max_hubs hubs and valid assignments."""

        hub_values = position[:self.num_nodes]
        hub_indices = np.argsort(hub_values)[-self.max_hubs:]
        hub_selection = list(hub_indices)
        hub_set = set(hub_selection)

        assignment_values = position[self.num_nodes:]
        spoke_assignments = {}

        for node in range(self.num_nodes):
            # A selected hub represents its own facility and therefore self-assigns.
            if node in hub_set:
                spoke_assignments[node] = node
                continue

            # Assignment genes are continuous; the nearest hub index decodes each gene.
            distances_to_hubs = [
                abs(assignment_values[node] - hub)
                for hub in hub_selection
            ]
            closest_hub = hub_selection[int(np.argmin(distances_to_hubs))]
            spoke_assignments[node] = closest_hub

        return hub_selection, spoke_assignments

    def calculate_distance(self, pos1, pos2):
        """Measure Euclidean distance in the encoded search space."""

        return np.linalg.norm(pos1 - pos2)

    def calculate_attractiveness(self, distance):
        """Decrease attraction exponentially with squared encoded distance."""

        return self.beta0 * np.exp(-self.gamma * distance**2)

    def move_firefly(self, firefly_i, firefly_j, distance):
        """Move a higher-cost firefly toward a lower-cost firefly with perturbation."""

        attractiveness = self.calculate_attractiveness(distance)
        random_walk = self.alpha_param * (
            np.random.random(len(firefly_i)) - 0.5
        )
        return (
            firefly_i
            + attractiveness * (firefly_j - firefly_i)
            + random_walk
        )

    def evaluate_solution(self, hub_selection, spoke_assignments):
        """Return fixed hub costs plus demand-weighted routing costs."""

        total_cost = sum(self.hub_costs[hub] for hub in hub_selection)

        for i in range(self.num_nodes):
            for j in range(self.num_nodes):
                if i == j:
                    continue

                demand = self.demand_matrix[i, j]
                if demand == 0:
                    continue

                hub_i = spoke_assignments[i]
                hub_j = spoke_assignments[j]

                # Only the inter-hub leg receives the model's transportation discount.
                route_distance = (
                    self.distance_matrix[i, hub_i]
                    + self.alpha * self.distance_matrix[hub_i, hub_j]
                    + self.distance_matrix[hub_j, j]
                )
                total_cost += (
                    route_distance * demand * self.transport_cost
                )

        return total_cost

    def initialize_population(self):
        """Create the initial population of encoded feasible networks."""

        population = []

        for _ in range(self.population_size):
            hub_selection = random.sample(
                range(self.num_nodes), self.max_hubs
            )
            spoke_assignments = {}

            for node in range(self.num_nodes):
                if node in hub_selection:
                    spoke_assignments[node] = node
                else:
                    spoke_assignments[node] = random.choice(hub_selection)

            population.append(
                self.encode_solution(hub_selection, spoke_assignments)
            )

        return np.array(population)

    def optimize(self, max_iterations=100):
        """Run the Firefly Algorithm and return the best decoded network found."""

        population = self.initialize_population()
        fitness_values = []

        for firefly in population:
            hub_selection, spoke_assignments = self.decode_solution(firefly)
            fitness_values.append(
                self.evaluate_solution(hub_selection, spoke_assignments)
            )

        fitness_values = np.array(fitness_values)
        best_idx = int(np.argmin(fitness_values))
        best_firefly = population[best_idx].copy()
        best_fitness = fitness_values[best_idx]
        convergence_history = [best_fitness]

        for _ in range(max_iterations):
            # Updates are sequential: later comparisons use the current population and costs.
            for i in range(self.population_size):
                for j in range(self.population_size):
                    if i == j or fitness_values[i] <= fitness_values[j]:
                        continue

                    distance = self.calculate_distance(
                        population[i], population[j]
                    )
                    population[i] = self.move_firefly(
                        population[i], population[j], distance
                    )

                    hub_selection, spoke_assignments = self.decode_solution(
                        population[i]
                    )
                    fitness_values[i] = self.evaluate_solution(
                        hub_selection, spoke_assignments
                    )

            current_best_idx = int(np.argmin(fitness_values))
            if fitness_values[current_best_idx] < best_fitness:
                best_fitness = fitness_values[current_best_idx]
                best_firefly = population[current_best_idx].copy()

            convergence_history.append(best_fitness)

        best_hub_selection, best_spoke_assignments = self.decode_solution(
            best_firefly
        )

        return {
            "hub_selection": best_hub_selection,
            "spoke_assignments": best_spoke_assignments,
            "total_cost": best_fitness,
            "convergence_history": convergence_history,
            "iterations": max_iterations,
        }

In [ ]:
def visualize_firefly_optimization(firefly_optimizer, result, coordinates):
    """Plot the network, search progress, utilization, and cost components."""

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(16, 12)
    )
    fig.suptitle(
        "Firefly Algorithm Hub-and-Spoke Network Optimization",
        fontsize=16,
        fontweight="bold",
    )

    best_hubs = result["hub_selection"]
    best_assignments = result["spoke_assignments"]

    # Panel 1: node-to-hub assignments in physical coordinate space.
    ax1.set_title("Hub-and-Spoke Assignments", fontweight="bold")
    for node, hub in best_assignments.items():
        if node != hub:
            ax1.plot(
                [coordinates[node, 0], coordinates[hub, 0]],
                [coordinates[node, 1], coordinates[hub, 1]],
                color="gray",
                alpha=0.3,
                linewidth=1,
            )

    colors = ["#FF6B6B", "#4ECDC4", "#FFD93D", "#95E77E", "#FFB6C1"]
    for node, (x_coord, y_coord) in enumerate(coordinates):
        if node in best_hubs:
            hub_idx = best_hubs.index(node)
            color = colors[hub_idx % len(colors)]
            ax1.scatter(
                x_coord,
                y_coord,
                s=400,
                c=color,
                marker="s",
                edgecolors="black",
                linewidth=2,
                label=f"Hub {node + 1}",
            )
        else:
            hub = best_assignments[node]
            hub_idx = best_hubs.index(hub)
            color = colors[hub_idx % len(colors)]
            ax1.scatter(
                x_coord,
                y_coord,
                s=200,
                c=color,
                alpha=0.7,
                edgecolors="black",
                linewidth=1,
            )
        ax1.text(x_coord + 0.1, y_coord + 0.1, f"{node + 1}")

    ax1.set_xlabel("X coordinate (coordinate units)")
    ax1.set_ylabel("Y coordinate (coordinate units)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

    # Panel 2: best-so-far objective value, including the initial population.
    ax2.set_title("Best-So-Far Cost", fontweight="bold")
    convergence = result["convergence_history"]
    ax2.plot(
        range(len(convergence)),
        convergence,
        "o-",
        linewidth=2,
        markersize=4,
        color="#FF6B6B",
    )
    ax2.set_xlabel("Iteration (0 is the initial population)")
    ax2.set_ylabel("Best-so-far cost (model units)")
    ax2.grid(True, alpha=0.3)

    # Panel 3: count all nodes, including each hub's self-assignment.
    ax3.set_title("Number of Nodes Assigned to Each Hub", fontweight="bold")
    hub_utilization = {
        hub: sum(
            assigned_hub == hub
            for assigned_hub in best_assignments.values()
        )
        for hub in best_hubs
    }
    hub_labels = [f"Hub {hub + 1}" for hub in best_hubs]
    utilization_values = [hub_utilization[hub] for hub in best_hubs]
    bars = ax3.bar(
        hub_labels,
        utilization_values,
        color=colors[:len(best_hubs)],
        alpha=0.8,
    )
    ax3.set_ylabel("Assigned nodes (count)")
    ax3.set_xlabel("Hub")
    ax3.grid(True, alpha=0.3)
    for bar, value in zip(bars, utilization_values):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            str(value),
            ha="center",
            va="bottom",
        )

    # Panel 4: separate fixed facility costs from routing costs.
    ax4.set_title("Cost Breakdown", fontweight="bold")
    fixed_cost = sum(
        firefly_optimizer.hub_costs[hub] for hub in best_hubs
    )
    variable_cost = result["total_cost"] - fixed_cost
    categories = ["Fixed hub costs", "Variable transport costs"]
    values = [fixed_cost, variable_cost]
    bars = ax4.bar(
        categories,
        values,
        color=["#FFB6C1", "#87CEEB"],
        alpha=0.8,
    )
    ax4.set_ylabel("Cost (model units)")
    ax4.grid(True, alpha=0.3)
    for bar, value in zip(bars, values):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            f"{value:,.0f}",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.show()

    print("\n=== FIREFLY ALGORITHM RESULTS ===")
    print(f"Total cost: {result['total_cost']:,.2f} model units")
    print(f"Number of hubs: {len(best_hubs)}")
    print(f"Iterations: {result['iterations']}")
    print(f"Hub locations (1-based labels): {[hub + 1 for hub in best_hubs]}")
    print(f"Fixed hub costs: {fixed_cost:,.2f} model units")
    print(f"Variable transport costs: {variable_cost:,.2f} model units")
    print("Nodes assigned to each hub:")
    for hub in best_hubs:
        print(f"  Hub {hub + 1}: {hub_utilization[hub]} nodes")

In [ ]:
# Build and run the specified 15-node example.
firefly_optimizer = FireflyHubSpokeOptimizer(
    num_nodes=15,
    max_hubs=5,
    population_size=20,
)

print("Running Firefly Algorithm optimization...")
start_time = time.time()
firefly_result = firefly_optimizer.optimize(max_iterations=50)
firefly_time = time.time() - start_time

print(f"Firefly Algorithm runtime: {firefly_time:.2f} seconds")
visualize_firefly_optimization(
    firefly_optimizer,
    firefly_result,
    firefly_optimizer.coordinates,
)

In [ ]:
def greedy_hub_spoke_solution(optimizer):
    """Build a reference solution using low fixed costs and nearest hubs."""

    hub_candidates = sorted(
        range(optimizer.num_nodes),
        key=lambda node: optimizer.hub_costs[node],
    )
    selected_hubs = hub_candidates[:optimizer.max_hubs]
    assignments = {}

    for node in range(optimizer.num_nodes):
        if node in selected_hubs:
            assignments[node] = node
        else:
            distances_to_hubs = [
                (optimizer.distance_matrix[node, hub], hub)
                for hub in selected_hubs
            ]
            assignments[node] = min(distances_to_hubs)[1]

    cost = optimizer.evaluate_solution(selected_hubs, assignments)
    return {
        "hub_selection": selected_hubs,
        "spoke_assignments": assignments,
        "total_cost": cost,
    }


# Compare costs on the same network data and objective function.
greedy_result = greedy_hub_spoke_solution(firefly_optimizer)
firefly_cost = firefly_result["total_cost"]
greedy_cost = greedy_result["total_cost"]

print("\n=== PERFORMANCE COMPARISON ===")
print(f"Firefly Algorithm: {firefly_cost:,.2f} model units")
print(f"Greedy heuristic: {greedy_cost:,.2f} model units")
if greedy_cost != 0:
    relative_difference = (greedy_cost - firefly_cost) / greedy_cost * 100
    print(
        "Firefly cost reduction relative to the greedy cost: "
        f"{relative_difference:.2f}%"
    )
else:
    print("Relative cost difference is undefined because the greedy cost is zero.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Firefly Algorithm and Greedy Heuristic Comparison",
    fontsize=16,
    fontweight="bold",
)

methods = ["Firefly Algorithm", "Greedy heuristic"]
costs = [firefly_cost, greedy_cost]
bars = ax1.bar(
    methods,
    costs,
    color=["#FF6B6B", "#4ECDC4"],
    alpha=0.8,
)
ax1.set_ylabel("Total cost (model units)")
ax1.set_title("Solution Cost Comparison")
ax1.grid(True, alpha=0.3)
for bar, cost in zip(bars, costs):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(costs) * 0.01,
        f"{cost:,.2f}",
        ha="center",
        va="bottom",
    )

ax2.set_title("Firefly Best-So-Far Convergence")
convergence = firefly_result["convergence_history"]
ax2.plot(
    range(len(convergence)),
    convergence,
    "o-",
    linewidth=2,
    markersize=4,
    color="#FF6B6B",
    label="Firefly best-so-far cost",
)
ax2.axhline(
    y=greedy_cost,
    color="#4ECDC4",
    linestyle="--",
    label=f"Greedy cost: {greedy_cost:,.2f}",
)
ax2.set_xlabel("Iteration (0 is the initial population)")
ax2.set_ylabel("Cost (model units)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def analyze_parameter_sensitivity():
    """Compare single-run costs across selected Firefly parameter values."""

    population_sizes = [10, 20, 30, 40]
    results_population = []

    for pop_size in population_sizes:
        optimizer = FireflyHubSpokeOptimizer(
            num_nodes=15,
            max_hubs=5,
            population_size=pop_size,
        )
        result = optimizer.optimize(max_iterations=30)
        results_population.append(result["total_cost"])

    gamma_values = [0.05, 0.1, 0.2, 0.3]
    results_gamma = []

    for gamma in gamma_values:
        optimizer = FireflyHubSpokeOptimizer(
            num_nodes=15,
            max_hubs=5,
            population_size=20,
        )
        optimizer.gamma = gamma
        result = optimizer.optimize(max_iterations=30)
        results_gamma.append(result["total_cost"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(
        "Firefly Algorithm Parameter Sensitivity",
        fontsize=16,
        fontweight="bold",
    )

    ax1.plot(
        population_sizes,
        results_population,
        "o-",
        linewidth=2,
        markersize=8,
        color="#FF6B6B",
    )
    ax1.set_xlabel("Population size (fireflies)")
    ax1.set_ylabel("Best cost found (model units)")
    ax1.set_title("Population Size Impact")
    ax1.grid(True, alpha=0.3)

    ax2.plot(
        gamma_values,
        results_gamma,
        "o-",
        linewidth=2,
        markersize=8,
        color="#4ECDC4",
    )
    ax2.set_xlabel("Gamma (light absorption coefficient)")
    ax2.set_ylabel("Best cost found (model units)")
    ax2.set_title("Light Absorption Impact")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\n=== PARAMETER SENSITIVITY ANALYSIS ===")
    print("Population size impact (single run per setting):")
    for pop_size, cost in zip(population_sizes, results_population):
        print(f"  Population size {pop_size}: {cost:,.2f} model units")

    print("Gamma impact (single run per setting):")
    for gamma, cost in zip(gamma_values, results_gamma):
        print(f"  Gamma {gamma}: {cost:,.2f} model units")


analyze_parameter_sensitivity()

## Interpretation and limitations

### What the method contributes

The population-based search maintains multiple candidate networks and uses cost comparisons to guide movements through an encoded solution space. Distance-dependent attraction encourages movement toward lower-cost candidates, while random perturbations provide exploration. The retained best-so-far solution makes progress visible through the convergence history, but a flat history does not prove that a global optimum has been reached.

The greedy heuristic provides a reference built from low fixed-cost hub choices and nearest-hub assignments. It is not an optimality benchmark: a lower Firefly cost on this instance is an observed comparison, not a general guarantee. The reported percentage is relative to the greedy cost and can be negative if the Firefly solution costs more.

### Reading the parameter experiment

The sensitivity cell varies population size and the light absorption coefficient `gamma` while keeping the network size, hub count, and iteration count specified in that cell. Each setting produces one stochastic search run, so differences reflect both parameter effects and run variability. The plots are evidence about these runs, not statistical estimates of average performance. A concept not evident in a particular run should be interpreted from its measured result rather than assumed to improve the solution.

### Practical limitations

- The continuous position vector is decoded into discrete hub and assignment decisions. Different positions can decode to the same network, and small movements need not change the decoded solution.
- The search parameters influence movement and exploration; a single run does not establish a universally suitable setting.
- The objective includes fixed hub costs and demand-weighted routing costs, but does not model capacity, congestion, or other operational constraints.
- The approximate search effort grows with the population size, iteration count, and number of origin-destination pairs. Larger instances may therefore require substantially more computation.

The method is useful when a heuristic search over candidate hub-and-spoke networks is appropriate and its results can be assessed against the stated objective and constraints. Its output should be interpreted as the best feasible network found during the specified run.